Import libraries and define the Dataset A path

In [1]:

# DATASET A AUDIT

# Purpose:
# This notebook is the first step of the IBY project.
# We will inspect Dataset A to understand:
#   1. How the data is organized
#   2. How sessions and chunks are structured
#   3. What information is contained in the event logs
#   4. What the ground-truth files look like
#
# Dataset A contains ground truth, so it will later be used
# to develop and validate our process-segmentation approach.


import json
from pathlib import Path
from collections import Counter

# Path to the combined Dataset A
dataset_a = Path("dataset_A") / "dataset_a_combined"

print("Dataset A path:", dataset_a)
print("Path exists:", dataset_a.exists())

Dataset A path: dataset_A\dataset_a_combined
Path exists: True


Find all sessions

In [2]:

# IDENTIFY SESSIONS

# Each session represents one recording period.
# A session can contain multiple chunks, so we must NOT
# treat every chunk as an independent session.


sessions = sorted([
    p for p in dataset_a.iterdir()
    if p.is_dir() and p.name.startswith("ses_")
])

print("Number of sessions:", len(sessions))

print("\nFirst 5 sessions:")
for session in sessions[:5]:
    print(" -", session.name)

Number of sessions: 63

First 5 sessions:
 - ses_20260630-121953-LAPTOP-R36BQBTE
 - ses_20260630-124826-CHAITANYA0BCF
 - ses_20260630-125757-LAPTOP-R36BQBTE
 - ses_20260630-131729-CHAITANYA0BCF
 - ses_20260630-132737-LAPTOP-R36BQBTE


Inspect the first session

In [3]:

# INSPECT ONE SESSION

# We begin with one session rather than loading the entire
# dataset. This lets us understand the data structure before
# designing the segmentation algorithm.


session = sessions[0]

print("Selected session:")
print(session.name)

print("\nContents of the session:")

for item in sorted(session.iterdir()):
    print(" -", item.name)

Selected session:
ses_20260630-121953-LAPTOP-R36BQBTE

Contents of the session:
 - chunk_1200
 - chunk_1230
 - chunk_20260630-1200-LAPTOP-R36BQBTE
 - chunk_20260630-1230-LAPTOP-R36BQBTE
 - gt.jsonl
 - gt_manifest.json


Find the event logs

In [4]:

# FIND EVENT LOGS

# A single session may be divided into multiple chunks.
# Each chunk contains an events.jsonl file.
#
# Therefore, we recursively search inside the session instead
# of assuming that there is only one events.jsonl file.


event_files = sorted(session.rglob("events.jsonl"))

print("Number of event-log files (chunks):", len(event_files))

print("\nEvent files:")

for file in event_files:
    print(" -", file)

Number of event-log files (chunks): 2

Event files:
 - dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1200-LAPTOP-R36BQBTE\events.jsonl
 - dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1230-LAPTOP-R36BQBTE\events.jsonl


Inspect the actual events

In [5]:

# INSPECT RAW EVENTS

# We read only the first 10 events.
#
# The purpose is NOT to analyze the whole dataset yet.
# We first need to discover the actual schema of the events:
# timestamps, event types, applications, mouse/keyboard data,
# screenshots, etc.
#
# We should not assume the field names before inspecting them.


event_file = event_files[0]

print("Inspecting:")
print(event_file)
print("\nFirst 10 events:\n")

with open(event_file, "r", encoding="utf-8") as f:

    for i in range(10):

        line = f.readline()

        if not line:
            break

        event = json.loads(line)

        print(f"--- Event {i + 1} ---")
        print(json.dumps(event, indent=2, ensure_ascii=False))

Inspecting:
dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1200-LAPTOP-R36BQBTE\events.jsonl

First 10 events:

--- Event 1 ---
{
  "schema_version": "1.0.0",
  "event_id": "evt_f29ff271-1691-4070-9b8f-02aeead78ff2",
  "session_id": "ses_20260630-121953-LAPTOP-R36BQBTE",
  "timestamp_ms": 1782821993821,
  "timestamp_iso": "2026-06-30T12:19:53.821Z",
  "layer": "SYSTEM",
  "event_type": "session_start",
  "source": {
    "agent_version": "1.1.1",
    "machine_id": "LAPTOP-R36BQBTE",
    "os": "Windows Windows_NT",
    "username_hash": "sha256:8c2ea5ba76c042f4"
  },
  "context": {
    "active_app": null,
    "active_monitor": null,
    "active_browser_tab": null,
    "visible_windows": null,
    "open_apps": null
  },
  "correlation": {
    "triggered_by": null,
    "correlated_events": [],
    "sequence_number": 0,
    "chunk_id": "chunk_20260630-1200-LAPTOP-R36BQBTE",
    "ms_since_last_event": null
  },
  "payload": {
    "agent_version": "1.1.1",
    

Inspect the ground truth

In [6]:

# INSPECT GROUND TRUTH

# Dataset A provides ground truth through gt.jsonl.
#
# This file tells us where business processes occurred.
# We will eventually use this information to evaluate how well
# our segmentation method recovers individual units of work.


gt_file = session / "gt.jsonl"

print("Ground-truth file:")
print(gt_file)

print("\nFirst 10 ground-truth records:\n")

with open(gt_file, "r", encoding="utf-8") as f:

    for i in range(10):

        line = f.readline()

        if not line:
            break

        gt = json.loads(line)

        print(f"--- Ground Truth {i + 1} ---")
        print(json.dumps(gt, indent=2, ensure_ascii=False))

Ground-truth file:
dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\gt.jsonl

First 10 ground-truth records:

--- Ground Truth 1 ---
{
  "ts_utc": "2026-06-30T12:20:25.140811+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "run_config",
  "seed": 645230,
  "dwell_scale": 1.4,
  "n_procs": 9,
  "run_date": "2026-06-30",
  "operator": "田中 健一",
  "operator_dept": "人事部",
  "machine_id": "NB-M1-05",
  "noise_rate": 0.2,
  "noise_blocks": 2,
  "chunk_split": "time:420",
  "continuation_family": "B",
  "continuation_dwell": 2.0,
  "split_min_gap": 6,
  "duration": "medium",
  "domain_mix": {},
  "transition": "random",
  "operator_type": "expert",
  "tasks_per_proc": [
    3,
    5
  ],
  "selected_procs": [
    "E",
    "C",
    "A",
    "B",
    "H",
    "J",
    "G",
    "O",
    "M"
  ],
  "split_procs": [
    "B"
  ],
  "session_notes": "田中 健一 — 9 procs — 2026-06-30 — seed 645230"
}
--- Ground Truth 2 ---
{
  "ts_utc": "2026-06-30T12:21:12.794418+00:00",
  "run

Count event types

In [ ]:

# INSPECT EVENT SCHEMA

# Before counting event types, we first inspect which keys
# are actually present in the event records.


event_keys = Counter()

with open(event_file, "r", encoding="utf-8") as f:

    for line in f:

        event = json.loads(line)

        event_keys.update(event.keys())

print("Event fields found:")
print()

for key, count in event_keys.most_common():
    print(f"{key}: {count}")

Event fields found:

schema_version: 1116
event_id: 1116
session_id: 1116
timestamp_ms: 1116
timestamp_iso: 1116
layer: 1116
event_type: 1116
source: 1116
context: 1116
correlation: 1116
payload: 1116
metadata: 1116
extensions: 1116


Basic event count

In [ ]:

# COUNT EVENTS IN THE SELECTED CHUNK


event_count = 0

with open(event_file, "r", encoding="utf-8") as f:

    for line in f:

        if line.strip():
            event_count += 1

print("Events in selected chunk:", event_count)

Events in selected chunk: 1116


ground-truth record type inventory

In [10]:
gt_event_types = Counter()

with open(gt_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        record = json.loads(line)
        record_type = record.get("event", "UNKNOWN")

        gt_event_types[record_type] += 1

print("Ground-truth record types:")
print("-" * 50)

for record_type, count in gt_event_types.most_common():
    print(f"{record_type:<35} {count:>8}")

print("-" * 50)
print("Total ground-truth records:", sum(gt_event_types.values()))

Ground-truth record types:
--------------------------------------------------
clipboard_paste                           50
task_started                              32
clipboard_copy                            32
process_started                           31
process_switched_out                      26
run_config                                 1
process_suspended                          1
process_resumed                            1
session_ended                              1
--------------------------------------------------
Total ground-truth records: 175


process-related ground-truth inspection

In [11]:

# CELL 10 — INSPECT PROCESS-RELATED GROUND TRUTH


process_events = []

with open(gt_file, "r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        if record.get("event") in {
            "process_started",
            "process_switched_out",
            "process_suspended",
            "process_resumed"
        }:
            process_events.append(record)

print("Number of process-related records:", len(process_events))

print("\nFirst 10 process-related records:\n")

for i, record in enumerate(process_events[:10], start=1):

    print(f"--- Process Record {i} ---")
    print(json.dumps(record, indent=2, ensure_ascii=False))

Number of process-related records: 59

First 10 process-related records:

--- Process Record 1 ---
{
  "ts_utc": "2026-06-30T12:21:12.794418+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_started",
  "current_process": "H",
  "process_code": "H",
  "process_name": "銀行勘定照合",
  "case_id": "BR-175009-001"
}
--- Process Record 2 ---
{
  "ts_utc": "2026-06-30T12:21:41.332221+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_switched_out",
  "current_process": "H",
  "from": "H",
  "to": "C"
}
--- Process Record 3 ---
{
  "ts_utc": "2026-06-30T12:21:41.335282+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_started",
  "current_process": "C",
  "process_code": "C",
  "process_name": "育児・産休申請確認",
  "case_id": "LA-175009-001"
}
--- Process Record 4 ---
{
  "ts_utc": "2026-06-30T12:22:25.680580+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_switched_out",
  "current_process": "C",
  "from": "C",
  "to": "B"
}
--- Pro